# QuantumHeal NH3 viewer companion
Exports full-precision molecular motions and vibrational results for the interactive report.

**No rerun is required to view the supplied HTML**: it already embeds a reproduced calculation. The submitted notebook remains unchanged.

Two ways to use this companion:
1. **Append route:** copy only the final Export cell into the end of your submitted notebook and run it after Section 9. This uses the live Qiskit VQE state and requires no new calculation.
2. **Standalone route:** run every cell here in Colab or a supported Python environment. This repeats the submitted vibrational workflow with the same Qiskit circuit; it does not rerun the electronic active-space calculation. Installation may require a kernel restart. This file deliberately does not pin an unverified new package release.

The fresh standalone runs may differ from saved submission values because optimizers and dependency versions change. The JSON export always computes its VQE error from the current Hamiltonian and current best warm run, avoiding stale display variables.


In [ ]:
%pip install -q "qiskit>=2.0,<3" "pyscf>=2.5,<3" numpy scipy matplotlib pyberny

In [ ]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
SEEDS = [7, 21, 42]
test_angles = np.array([0.2, 0.3, -0.4])
NH3_GEOMETRY = """N 0 0 0
H 0 0 1.0100
H .9530 0 -.3370
H -.4765 .8253 -.3370"""
def make_ansatz(theta):
    a,b,c = map(float,theta)
    qc = QuantumCircuit(2)
    qc.ry(a,0); qc.ry(b,1); qc.cry(c,0,1)
    return qc


## 1 Optimize geometry and calculate the six modes

In [ ]:
# Optimize the NH3 geometry and compute its harmonic vibrational modes.
from pyscf import gto, scf
from pyscf.data import nist
from pyscf.geomopt.berny_solver import optimize
from pyscf.hessian import thermo
from qiskit.quantum_info import SparsePauliOp, Operator

HA_TO_CM = nist.HARTREE2WAVENUMBER   # 1 Hartree = 219474.63 cm^-1
BOHR_TO_ANG = nist.BOHR              # 1 Bohr = 0.529177 Angstrom

# Same molecule, basis and method (RHF/STO-3G) as Sections 1-8, starting from the same coordinates.
mol_start = gto.M(
    atom=";".join(line.strip() for line in NH3_GEOMETRY.strip().splitlines()),
    basis="sto3g", charge=0, spin=0, unit="Angstrom", verbose=0,
)
mf_start = scf.RHF(mol_start).run()

# Vibrations are defined around an energy minimum, so optimize the geometry first.
mol_eq = optimize(mf_start, maxsteps=50)
mf_eq = scf.RHF(mol_eq).run()
E_eq = mf_eq.e_tot

# Hessian -> harmonic frequencies (cm^-1) and normal-mode displacement vectors.
hessian = mf_eq.Hessian().kernel()
harmonic = thermo.harmonic_analysis(mol_eq, hessian)
freqs_cm = np.asarray(harmonic["freq_wavenumber"])
assert not np.iscomplexobj(freqs_cm) and np.all(freqs_cm > 0), "Imaginary frequency: not a true minimum."

coords_eq = mol_eq.atom_coords()     # Bohr; atom 0 is N, atoms 1-3 are H

def nh_bond_lengths(coords):
    # The three N-H distances (same units as coords).
    return np.array([np.linalg.norm(coords[i] - coords[0]) for i in (1, 2, 3)])

r_eq = nh_bond_lengths(coords_eq)

# How much does each N-H bond change when we move a small step along each mode?
step = 0.01
bond_response = np.array([(nh_bond_lengths(coords_eq + step * m) - r_eq) / step
                          for m in harmonic["norm_mode"]])

# Symmetric stretch: all three bonds change by the same amount, and strongly.
is_symmetric = np.ptp(bond_response, axis=1) < 0.05 * np.abs(bond_response).max(axis=1)
k_stretch = int(np.argmax(np.where(is_symmetric, np.abs(bond_response.mean(axis=1)), 0.0)))
mode_vec = harmonic["norm_mode"][k_stretch]
if bond_response[k_stretch].mean() < 0:
    mode_vec = -mode_vec             # convention: positive displacement = longer N-H bonds
    bond_response[k_stretch] *= -1
omega_cm = float(freqs_cm[k_stretch])
omega_au = omega_cm / HA_TO_CM       # harmonic frequency in Hartree (hbar = 1)

print(f"Fixed starting geometry energy (Sections 1-8): {mf_start.e_tot:.8f} Ha")
print(f"Optimized geometry energy:                     {E_eq:.8f} Ha")
print(f"  Lowered by {1000 * (mf_start.e_tot - E_eq):.3f} mHa; optimized N-H length = {r_eq[0] * BOHR_TO_ANG:.4f} A\n")
print(f"{'Mode':>4} {'Harmonic freq (cm^-1)':>22}   N-H bond changes (relative)")
for k, (f, d) in enumerate(zip(freqs_cm, bond_response)):
    tag = "  <- symmetric N-H stretch (selected)" if k == k_stretch else ""
    print(f"{k:>4} {f:>22.1f}   {np.round(d, 3)}{tag}")


## 2 Scan the symmetric stretch with classical RHF

In [ ]:
#  Scan the RHF energy along the symmetric-stretch normal mode.
def geometry_at(xi):
    # Cartesian coordinates (Bohr) displaced by the dimensionless normal coordinate xi.
    # xi = sqrt(omega) * q in atomic units, and PySCF's normal modes use mass in amu,
    # so the mass-weighted displacement is Q = xi / sqrt(omega_au * amu_in_electron_masses).
    Q = xi / np.sqrt(omega_au * nist.AMU2AU)
    return coords_eq + Q * mode_vec

xi_scan = np.linspace(-3.5, 3.5, 15)
E_scan = []
dm_guess = mf_eq.make_rdm1()
for xi in xi_scan:
    mol_xi = mol_eq.copy()
    mol_xi.set_geom_(geometry_at(xi), unit="Bohr")
    mf_xi = scf.RHF(mol_xi)
    mf_xi.conv_tol = 1e-11
    mf_xi.kernel(dm0=dm_guess)       # start from the equilibrium density for smooth convergence
    assert mf_xi.converged, f"SCF did not converge at xi = {xi}"
    E_scan.append(mf_xi.e_tot)

V_scan = np.array(E_scan) - E_eq     # potential relative to the minimum (Hartree)
r_scan = np.array([nh_bond_lengths(geometry_at(xi))[0] for xi in xi_scan]) * BOHR_TO_ANG

# 6th-order polynomial fit of V(xi).
POLY_DEGREE = 6
poly_coeffs = np.polyfit(xi_scan, V_scan, POLY_DEGREE)
fit_residual_cm = np.abs(V_scan - np.polyval(poly_coeffs, xi_scan)).max() * HA_TO_CM
assert poly_coeffs[0] > 0, "Fitted potential must rise at large |xi|."

# Curvature check: V = 1/2 omega xi^2 near the minimum, so omega_fit = 2 * (xi^2 coefficient).
omega_fit_cm = 2 * poly_coeffs[-3] * HA_TO_CM

print(f"Scanned N-H bond lengths: {r_scan.min():.3f} to {r_scan.max():.3f} A (equilibrium {r_eq[0] * BOHR_TO_ANG:.3f} A)")
print(f"{'xi':>6} {'N-H (A)':>8} {'V (cm^-1)':>11}")
for xi, r, v in zip(xi_scan, r_scan, V_scan * HA_TO_CM):
    print(f"{xi:>6.2f} {r:>8.4f} {v:>11.1f}")
print(f"\nLargest fit residual: {fit_residual_cm:.2f} cm^-1")
print(f"Frequency from fit curvature: {omega_fit_cm:.1f} cm^-1 vs Hessian: {omega_cm:.1f} cm^-1")


## 3 Build the vibrational Hamiltonian

In [ ]:
# Vibrational Hamiltonian in the harmonic-oscillator basis, mapped to 2 qubits.
N_LEVELS = 4    # n = 0, 1, 2, 3  ->  2 qubits
N_BIG = 60      # large basis, used only for the converged classical reference

def vibrational_hamiltonian(n_basis):
    # H = (omega/2) p^2 + V(xi), with xi and p written using ladder operators (Hartree).
    a = np.diag(np.sqrt(np.arange(1, n_basis)), k=1)   # lowering operator
    X = (a + a.T) / np.sqrt(2)                          # position xi
    P2 = -((a.T - a) @ (a.T - a)) / 2                   # momentum squared
    V = sum(c * np.linalg.matrix_power(X, p) for p, c in enumerate(poly_coeffs[::-1]))
    return 0.5 * omega_au * P2 + V

H_big = vibrational_hamiltonian(N_BIG)
# Taking the 4 x 4 corner of the large-basis matrix gives exact matrix elements of the
# polynomial potential between the 4 lowest harmonic-oscillator states.
H_vib_matrix = H_big[:N_LEVELS, :N_LEVELS]
assert np.allclose(H_vib_matrix, H_vib_matrix.T), "Vibrational Hamiltonian must be symmetric."

# Binary encoding: level n -> 2-qubit basis state |n>.
H_vib = SparsePauliOp.from_operator(Operator(H_vib_matrix)).simplify()
assert H_vib.num_qubits == 2

E_vib_exact = np.linalg.eigvalsh(H_vib_matrix)           # same 4-level model as VQE
E_vib_converged = np.linalg.eigvalsh(H_big)[:N_LEVELS]   # truncation check
E_vib_harmonic = (np.arange(N_LEVELS) + 0.5) * omega_au  # simple harmonic model

print("4 x 4 vibrational Hamiltonian (cm^-1):")
print(np.round(H_vib_matrix * HA_TO_CM, 1))
print(f"\nQubit Hamiltonian: {len(H_vib)} Pauli terms (coefficients in cm^-1)")
for pauli, coefficient in zip(H_vib.paulis.to_labels(), H_vib.coeffs):
    print(f"  {coefficient.real * HA_TO_CM:+10.2f} x {pauli}")
print(f"\n{'n':>2} {'Harmonic':>10} {'Exact 4-level':>14} {'Converged':>10}   (cm^-1)")
for n in range(N_LEVELS):
    print(f"{n:>2} {E_vib_harmonic[n] * HA_TO_CM:>10.1f} {E_vib_exact[n] * HA_TO_CM:>14.1f} "
          f"{E_vib_converged[n] * HA_TO_CM:>10.1f}")
print(f"\nFundamental transition (n=0 -> 1), converged: "
      f"{(E_vib_converged[1] - E_vib_converged[0]) * HA_TO_CM:.1f} cm^-1 (harmonic: {omega_cm:.1f})")


## 4 Run Qiskit statevector VQE

In [ ]:
#  VQE for the vibrational ground state, using the same circuit as before.
def vib_energy(theta):
    # Exact statevector estimate of <psi(theta)|H_vib|psi(theta)> in Hartree.
    return float(np.real(Statevector(make_ansatz(theta)).expectation_value(H_vib)))

# Cross-check Qiskit against direct matrix multiplication (as in Section 5).
psi_test = Statevector(make_ansatz(test_angles)).data
assert np.isclose(vib_energy(test_angles), np.real(np.vdot(psi_test, H_vib_matrix @ psi_test)), atol=1e-12)

MAX_EVAL_VIB = 300

def run_vib_vqe(x0, rhobeg):
    history = []
    def objective(theta):
        energy = vib_energy(theta)
        history.append(energy)
        return energy
    result = minimize(objective, x0=x0, method="COBYLA",
                      options={"maxiter": MAX_EVAL_VIB, "rhobeg": rhobeg, "tol": 1e-10})
    return result, np.array(history)

vib_runs = []
for start, low, high, rhobeg in [("warm", -1.0, 1.0, 0.3), ("cold", -np.pi, np.pi, 0.6)]:
    for seed in SEEDS:
        x0 = np.random.default_rng(seed).uniform(low, high, size=3)
        result, history = run_vib_vqe(x0, rhobeg)
        error_cm = (result.fun - E_vib_exact[0]) * HA_TO_CM
        vib_runs.append({"start": start, "seed": seed, "result": result, "history": history, "error_cm": error_cm})
        print(f"{start} start, seed {seed:>2}: E = {result.fun * HA_TO_CM:.4f} cm^-1; error = {error_cm:.4f} cm^-1")

best_vib = min((r for r in vib_runs if r["start"] == "warm"), key=lambda r: r["result"].fun)
E_vib_vqe = vib_energy(best_vib["result"].x)
vib_error_cm = (E_vib_vqe - E_vib_exact[0]) * HA_TO_CM

print(f"\nBest warm-start seed: {best_vib['seed']}")
print(f"VQE zero-point energy (this mode):  {E_vib_vqe * HA_TO_CM:.4f} cm^-1")
print(f"Exact 4-level zero-point energy:    {E_vib_exact[0] * HA_TO_CM:.4f} cm^-1")
print(f"VQE error:                          {vib_error_cm:.4f} cm^-1")
probs = np.abs(Statevector(make_ansatz(best_vib["result"].x)).data) ** 2
print("Level populations in the VQE state:", ", ".join(f"n={n}: {p:.4f}" for n, p in enumerate(probs)))


## 5 Export cell — can be appended to the submitted notebook
Run after the original Section 9 (or after the standalone cells above). Import the resulting JSON through the HTML report's **Import calculation JSON** control. The report accepts one four-atom geometry, six Cartesian displacement vectors, a 15-point scan, and the symmetric-stretch model. Modes use arbitrary sign conventions; either sign is physically equivalent.

This exports harmonic mode vectors, not quantum nuclear trajectories. Excited vibrational levels and probability curves are from exact diagonalization of the same four-state Hamiltonian; the VQE energy is a separate ground-state result. The optional XYZ trajectory is for JSmol and depicts the selected symmetric stretch only.


In [ ]:
import json
from pathlib import Path
import numpy as np
from qiskit.quantum_info import Statevector

required = ['coords_eq','harmonic','freqs_cm','poly_coeffs','V_scan',
            'H_vib_matrix','H_big','vib_runs','k_stretch','mode_vec',
            'HA_TO_CM','BOHR_TO_ANG','make_ansatz']
missing = [key for key in required if key not in globals()]
if missing:
    raise RuntimeError('Run Section 9 of the original notebook first. Missing: '+', '.join(missing))
assert np.asarray(coords_eq).shape == (4,3)
assert np.asarray(harmonic['norm_mode']).shape == (6,4,3)
assert np.all(np.real(freqs_cm)>0) and np.max(np.abs(np.imag(freqs_cm))) < 1e-8
assert np.allclose(H_vib_matrix,H_vib_matrix.T.conj(),atol=1e-12)
vals, vectors = np.linalg.eigh(H_vib_matrix)
best = min((r for r in vib_runs if r['start']=='warm'), key=lambda r: r['result'].fun)
psi = Statevector(make_ansatz(best['result'].x)).data
vqe_ha = float(np.real(np.vdot(psi,H_vib_matrix @ psi)))
error_cm = (vqe_ha-vals[0])*HA_TO_CM
assert error_cm >= -1e-6
mode_vectors = np.asarray(harmonic['norm_mode']).real.copy()
mode_vectors[k_stretch] = mode_vec
payload = {
 'coords': (np.asarray(coords_eq)*BOHR_TO_ANG).tolist(),
 'vectors': mode_vectors.tolist(),
 'freqs': np.asarray(freqs_cm).real.tolist(),
 'stretch': int(k_stretch),
 'coeff': (np.asarray(poly_coeffs)*HA_TO_CM).tolist(),
 'scan': (np.asarray(V_scan)*HA_TO_CM).tolist(),
 'levels': (vals*HA_TO_CM).tolist(),
 'large': (np.linalg.eigvalsh(H_big)[:4]*HA_TO_CM).tolist(),
 'states': vectors.T.real.tolist(),
 'vqe': vqe_ha*HA_TO_CM,
 'provenance': 'computed',
 'source_note': 'Live kernel export; RHF/STO-3G potential; Qiskit noiseless statevector VQE.',
 'vqe_error_cm': float(error_cm),
 'best_seed': int(best['seed']),
}
assert len(payload['scan'])==15 and len(payload['coeff'])==7
path=Path('nh3_viewer_data.json')
path.write_text(json.dumps(payload,indent=2,allow_nan=False))
print('Exported',path.resolve())
print('Live VQE ZPE / cm^-1:',payload['vqe'])
print('Exact same-model ZPE / cm^-1:',payload['levels'][0])
print('VQE optimization error / cm^-1:',error_cm)
print('Fundamental, four-state model / cm^-1:',payload['levels'][1]-payload['levels'][0])
print('Fundamental, larger model / cm^-1:',payload['large'][1]-payload['large'][0])
# XYZ motion: physical normal-coordinate convention from the submitted notebook.
from pyscf.data import nist
omega = float(np.real(freqs_cm[k_stretch])) / HA_TO_CM
frames=[]
for f in range(40):
    xi=2.0*np.cos(2*np.pi*f/40)
    xyz=(coords_eq+xi/np.sqrt(omega*nist.AMU2AU)*mode_vec)*BOHR_TO_ANG
    frames.extend(['4',f'NH3 calculated symmetric stretch, xi={xi:.8f}'])
    frames.extend(('N' if i==0 else 'H')+' '+' '.join(f'{x:.9f}' for x in row)
                  for i,row in enumerate(xyz))
Path('NH3_symmetric_stretch.xyz').write_text('\n'.join(frames)+'\n')
try:
    from google.colab import files
    files.download(str(path))
except ImportError:
    print('Download the JSON through your notebook file browser.')


## Interpretation and next calculations
The six harmonic frequencies and potential scan are classical RHF results. The symmetric-stretch VQE solves the lowest energy of a four-state nuclear Hamiltonian. It does not produce all six frequencies or the excited-state transition by itself.

For VQE excited states, add an appropriate orthogonality-constrained solver (for example, variational quantum deflation), validate against the same matrix, and compare E1−E0. For other modes, scan and validate each potential separately. The umbrella inversion needs a double-well treatment; a local harmonic mode does not capture tunneling. For thermodynamics, converge enough levels for the temperature range, add the remaining molecular degrees of freedom, and keep ZPE reference conventions consistent.

The report's embedded reproduction uses the same circuit amplitudes evaluated analytically, with a larger optimizer budget. This notebook uses Qiskit statevectors as in the submission. Neither route demonstrates hardware execution or speedup.
